# NASCTN CBRS SEA: Grant List Size Analysis
This notebook plots and saves CBSD grant lists.

# Disclaimer
This software was developed by employees of the National Institute
of Standards and Technology (NIST), an agency of the Federal
Government. Pursuant to title 17 United States Code Section 105, works
of NIST employees are not subject to copyright protection in the United
States and are considered to be in the public domain. Permission to freely
use, copy, modify, and distribute this software and its documentation
without fee is hereby granted, provided that this notice and disclaimer
of warranty appears in all copies.

THE SOFTWARE IS PROVIDED 'AS IS' WITHOUT ANY WARRANTY OF ANY KIND,
EITHER EXPRESSED, IMPLIED, OR STATUTORY, INCLUDING, BUT NOT LIMITED
TO, ANY WARRANTY THAT THE SOFTWARE WILL CONFORM TO SPECIFICATIONS, ANY
IMPLIED WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE,
AND FREEDOM FROM INFRINGEMENT, AND ANY WARRANTY THAT THE DOCUMENTATION
WILL CONFORM TO THE SOFTWARE, OR ANY WARRANTY THAT THE SOFTWARE WILL BE
ERROR FREE. IN NO EVENT SHALL NIST BE LIABLE FOR ANY DAMAGES, INCLUDING,
BUT NOT LIMITED TO, DIRECT, INDIRECT, SPECIAL OR CONSEQUENTIAL DAMAGES,
ARISING OUT OF, RESULTING FROM, OR IN ANY WAY CONNECTED WITH THIS
SOFTWARE, WHETHER OR NOT BASED UPON WARRANTY, CONTRACT, TORT, OR
OTHERWISE, WHETHER OR NOT INJURY WAS SUSTAINED BY PERSONS OR PROPERTY
OR OTHERWISE, AND WHETHER OR NOT LOSS WAS SUSTAINED FROM, OR AROSE OUT
OF THE RESULTS OF, OR USE OF, THE SOFTWARE OR SERVICES PROVIDED HEREUNDER.

Distributions of NIST software should also include copyright and licensing
statements of any third-party software that are legally bundled with
the code in compliance with the conditions of those licenses.

In [51]:
import os
import csv
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import sys

current_dir = Path.cwd()
main_dir = Path(current_dir).parents[0]
src_dir = main_dir / 'src'
sys.path.append(os.path.abspath(src_dir))


In [52]:
# Configuration: DPA name and channels
dpa_name = "West14" # Change as needed (e.g., East1, West13, West14)
channels_start = list(range(3550, 3650, 10))
channels_end = list(range(3560, 3660, 10))
dpa_channels = list(zip(channels_start, channels_end))

# Setup directories
results_dir = main_dir / 'outputs' / 'results' / dpa_name
plots_dir = main_dir / 'outputs' / 'plots' / dpa_name
os.makedirs(plots_dir, exist_ok=True)


In [53]:
# Initialize lists for tracking sizes
channels = []
neighborListSizes = []
moveListSizes = []
keepListSizes = []
csv_rows = []

csv_fields = ['channel', 'center_frequency_mhz', 'neighbor_list_size', 'move_list_size', 'keep_list_size', 'global_list_size', 'global_keep_list_size']

# Process data
for ch, dpa_channel in enumerate(dpa_channels):
    
    dpa_channel_str = f'channel{dpa_channel[0]}to{dpa_channel[1]}'
    fc_mhz = int(sum(dpa_channel) / len(dpa_channel))
    
    # Load data
    fname = results_dir / f'{dpa_name}_{dpa_channel_str}_grantLists.npy'
    
    try:
        (protected_points, dpa_channel_data, neighborList, moveList, keepList, globalList, globalKeepList) = np.load(fname, allow_pickle=True)
         
        # Track metrics for plotting
        channels.append(str(ch+1))
        neighborListSizes.append(len(neighborList))
        moveListSizes.append(len(moveList))
        keepListSizes.append(len(keepList)) 
        
        # Track rows for CSV
        csv_rows.append([str(ch+1), fc_mhz, len(neighborList), len(moveList), len(keepList), len(globalList), len(globalKeepList)])
        
    except FileNotFoundError:
        print(f"Warning: File {fname} not found. Skipping channel {dpa_channel}.")

In [54]:
# Export data to CSV
if csv_rows:
    output_fname = results_dir / f'{dpa_name}_grantListSizes.csv'
    with open(output_fname, 'w', newline="") as csvfile:
        csvwriter = csv.writer(csvfile, quoting=csv.QUOTE_ALL)
        csvwriter.writerow(csv_fields)
        csvwriter.writerows(csv_rows)
    # print(f"\nGrant list sizes saved to CSV: {output_fname}")
else:
    print("No data processed; CSV export skipped.")

In [55]:
# Generate Visualization
if channels:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Create stacked bar chart
    ax.bar(channels, moveListSizes, color='C1', label="Move list")
    ax.bar(channels, keepListSizes, color='C1', alpha=0.4, bottom=moveListSizes, label="Keep list")
    
    # Add total size annotations above bars
    for i, total in enumerate(neighborListSizes):
        ax.text(i, total + .01 * max(neighborListSizes), round(total),
              ha='center', weight='bold', color='black', size=10)
    
    # Add inner bar annotations
    for bar in ax.patches:
        height = bar.get_height()
        if height > 0: # Only label if height is significant
            ax.text(bar.get_x() + bar.get_width() / 2,
                  height / 2 + bar.get_y(),
                  round(height), ha='center',
                  color='w', weight='bold', size=10)
            
    handles, labels = plt.gca().get_legend_handles_labels()
    if len(handles) > 1:
        order = [1, 0]
        plt.legend([handles[idx] for idx in order], [labels[idx] for idx in order], loc='upper left')
    
    xtick_values = list(range(channels_start[0]+5, channels_start[-1]+15, 10))
    xtick_labels = [str(item) for item in xtick_values]
    ax.set_xticks(range(len(channels)))
    ax.set_xticklabels(xtick_labels[:len(channels)], rotation=45)
    
    ax.set_xlabel("Center frequency (MHz)")
    ax.set_ylabel("Number of CBSD grants")
    ax.set_title(f"Grant List Sizes for {dpa_name}")
    plt.tight_layout()
    
    plot_fname = plots_dir / f'{dpa_name}_grantListSizes.png'
    plt.savefig(plot_fname)
    # plt.show()
    plt.close()
else:
    print("No data available to plot.")